# **Scail2 for Animation Transfer and Character Swap(ComfyUI)**
- Run the cell below to get a link (e.g. https://literature-consortium-align.trycloudflare.com ) which you can use to launch the comfyUI interface.
- You can get workflows here: https://comfyui.nomadoor.net/en/basic-workflows/scail-2/ (If you see a red node, change the model in the node to the one downloaded by this notebook.)
- Github project page: https://github.com/zai-org/SCAIL-2
- Notebook source: https://github.com/Isi-dev/Google-Colab_Notebooks

In [2]:
# @title # 💥Prepare Environment and get link to launch ComfyUI {"single-column":true}
import os

# 1. ALWAYS force the directory back to the Colab root before cloning
%cd /content

# 2. Clone ComfyUI only if it doesn't already exist in the root
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

# 3. Move into the correct, top-level ComfyUI folder
%cd /content/ComfyUI

# Install core dependencies
!pip install -r requirements.txt

# 4. Install custom nodes only if they don't already exist
# if not os.path.exists("custom_nodes/ComfyUI-Manager"):
#     !cd custom_nodes && git clone https://github.com/ltdrdata/ComfyUI-Manager.git

if not os.path.exists("custom_nodes/ComfyUI-GGUF"):
    !cd custom_nodes && git clone https://github.com/city96/ComfyUI-GGUF.git

# 5. Install Video Helper Suite
if not os.path.exists("custom_nodes/ComfyUI-VideoHelperSuite"):
    !cd custom_nodes && git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
    !cd custom_nodes/ComfyUI-VideoHelperSuite && pip install -r requirements.txt

# 6. Install KJNodes
if not os.path.exists("custom_nodes/ComfyUI-KJNodes"):
    !cd custom_nodes && git clone https://github.com/kijai/ComfyUI-KJNodes.git
    !cd custom_nodes/ComfyUI-KJNodes && pip install -r requirements.txt

# Install standard gguf dependency
!pip install gguf


# @title 2. Accelerated Model Downloader (Aria2c & Civitai)
# @markdown Paste your direct download links below. Ensure your Civitai Token is pasted if fetching gated models.

civitai_token = "" # @param {type:"string"}

# @markdown ---
# @markdown ### **Main SCAIL-2 Models**
# @markdown *Check the boxes for the model formats you wish to download.*
download_gguf = False # @param {type:"boolean"}
gguf_models = "" # @param {type:"string"}

download_safetensors = True # @param {type:"boolean"}
scail_safetensors = "https://huggingface.co/Comfy-Org/SCAIL-2/resolve/main/diffusion_models/wan2.1_14B_SCAIL_2_fp8_scaled.safetensors" # @param {type:"string"}

# @markdown ---
# @markdown ### **Required Models**
checkpoints = "https://huggingface.co/Comfy-Org/sam3.1/resolve/main/checkpoints/sam3.1_multiplex_fp16.safetensors" # @param {type:"string"}
text_encoders = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors" # @param {type:"string"}
clip_vision = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors" # @param {type:"string"}
vae = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors" # @param {type:"string"}

# @markdown ---
# @markdown ### **LoRA Models**
# @markdown *Paste up to 5 LoRAs. Leave unused boxes blank.*
lora_1 = "https://huggingface.co/lightx2v/Wan2.1-I2V-14B-480P-StepDistill-CfgDistill-Lightx2v/resolve/main/loras/Wan21_I2V_14B_lightx2v_cfg_step_distill_lora_rank64.safetensors" # @param {type:"string"}
lora_2 = "https://huggingface.co/Comfy-Org/SCAIL-2/resolve/main/loras/wan2.1_SCAIL_2_DPO_lora_bf16.safetensors" # @param {type:"string"}
lora_3 = "" # @param {type:"string"}
lora_4 = "" # @param {type:"string"}
lora_5 = "" # @param {type:"string"}

import os
import time
import subprocess
from pathlib import Path

%cd /content/ComfyUI

# --- SYSTEM SETUP ---
def install_apt_packages():
    try:
        subprocess.run(['apt-get', '-y', 'install', '-qq', 'aria2'], check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        print(f"✗ Error installing aria2: {e.stderr.decode().strip() or 'Unknown error'}")

# --- DOWNLOAD ENGINES ---
def download_civitai_model(url, dest_dir, token):
    os.makedirs(dest_dir, exist_ok=True)
    try:
        model_id = url.split("/models/")[1].split("?")[0]
    except IndexError:
        print(f"❌ Invalid Civitai URL format: {url}")
        return False

    civitai_url = f"https://civitai.com/api/download/models/{model_id}?type=Model&format=SafeTensor"
    if token:
        civitai_url += f"&token={token}"

    filename = f"civitai_model_{model_id}_{time.strftime('%Y%m%d_%H%M%S')}.safetensors"
    full_path = os.path.join(dest_dir, filename)

    print(f"Downloading Civitai model {model_id}...")
    download_command = f"wget -q --max-redirect=10 --show-progress \"{civitai_url}\" -O \"{full_path}\""
    os.system(download_command)

    if os.path.exists(full_path) and os.path.getsize(full_path) > 0:
        print(f"✅ Civitai model downloaded: {filename}\n")
        return True
    else:
        print(f"❌ Civitai download failed: {filename}\n")
        return False

def model_download_aria2c(url, dest_dir):
    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    filename = url.split('/')[-1].split('?')[0]
    file_path = os.path.join(dest_dir, filename)

    if os.path.exists(file_path):
        print(f"⏭️ Skipping {filename} (Already exists in {dest_dir})\n")
        return True

    cmd = [
        'aria2c', '--console-log-level=error', '-c', '-x', '16', '-s', '16', '-k', '1M',
        '-d', dest_dir, '-o', filename, url
    ]

    try:
        print(f"Downloading {filename} with aria2c...")
        subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"✅ Successfully downloaded {filename}\n")
        return True
    except subprocess.CalledProcessError as e:
        error = e.stderr.strip() or "Unknown error"
        print(f"❌ Error downloading {filename}: {error}\n")
        return False

# --- EXECUTION LOGIC ---
print("Installing aria2c...\n")
install_apt_packages()
print("Initializing download queue...\n")

token = civitai_token.strip()

# Conditionally compile the download lists based on the checkboxes
unet_downloads = []
if download_gguf:
    unet_downloads.append(gguf_models)

diffusion_downloads = []
if download_safetensors:
    diffusion_downloads.append(scail_safetensors)

# Map the UI fields to their correct folders as lists
download_map = {
    "models/unet": unet_downloads,
    "models/diffusion_models": diffusion_downloads,
    "models/checkpoints": [checkpoints],
    "models/text_encoders": [text_encoders],
    "models/clip_vision": [clip_vision],
    "models/vae": [vae],
    "models/loras": [lora_1, lora_2, lora_3, lora_4, lora_5]
}

for folder, url_list in download_map.items():
    for url in url_list:
        url = url.strip()

        # Skip empty boxes
        if not url:
            continue

        if "civitai.com" in url.lower():
            if not token:
                print(f"❌ Skipping Civitai link (No token provided): {url[:50]}...\n")
                continue
            download_civitai_model(url, folder, token)
        else:
            model_download_aria2c(url, folder)

print("🎉 All tasks processed!")

# @title 3. Run ComfyUI
use_cloudflare = True # @param {type:"boolean"}
use_interface_in_cell = False # @param {type:"boolean"}

import torch
import os
from IPython.display import clear_output

clear_output()

# Guarantee we are in the correct directory before launching
%cd /content/ComfyUI

# Dynamically check if you are on a T4 GPU to apply memory optimizations
using_T4_GPU = torch.cuda.get_device_name(0) == 'Tesla T4' if torch.cuda.is_available() else False

# Optimization for SCAIL-2 + Fix for Colab 403 Proxy Error
launch_args = "--highvram --enable-cors-header" if using_T4_GPU else "--enable-cors-header"

if use_cloudflare:
    # Only download cloudflared if we don't already have it
    if not os.path.exists("cloudflared-linux-amd64.deb"):
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
        !dpkg -i cloudflared-linux-amd64.deb

    import subprocess
    import threading
    import time
    import socket
    import urllib.request

    def iframe_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', port))
            if result == 0:
                break
            sock.close()
        print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

        p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        for line in p.stderr:
            l = line.decode()
            if "trycloudflare.com " in l:
                print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
        clear_output()

    threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

    !python main.py $launch_args

elif use_interface_in_cell:
    import threading
    import time
    import socket

    def iframe_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', port))
            if result == 0:
                break
            sock.close()
        from google.colab import output
        output.serve_kernel_port_as_iframe(port, height=1024)
        clear_output()
        print("To open it in a window you can open this link here:")
        output.serve_kernel_port_as_window(port)

    threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

    !python main.py $launch_args

else:
    import socket, time, threading
    from google.colab import output

    def link_thread(port):
        while True:
            time.sleep(0.5)
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            result = sock.connect_ex(('127.0.0.1', port))
            if result == 0:
                break
            sock.close()
        clear_output()
        print("Click the link below to launch the ComfyUI interface")
        output.serve_kernel_port_as_window(port)

    # Start thread for port 8188
    threading.Thread(target=link_thread, daemon=True, args=(8188,)).start()

    !python main.py $launch_args